In [4]:
# Importacion de librerias 
import pandas as pd
import matplotlib.pyplot as plt
df = pd.read_csv("server_logs.csv") # Lectura del csv utilizando la funcion de pandas .read_csv()
df.info()                           # Imprime toda la informacion del DataFrame (columnas, tipo de datos, uso de memoria)

<class 'pandas.DataFrame'>
RangeIndex: 5795 entries, 0 to 5794
Data columns (total 15 columns):
 #   Column           Non-Null Count  Dtype
---  ------           --------------  -----
 0   timestamp_event  5795 non-null   str  
 1   received_at      5795 non-null   str  
 2   service_name     5795 non-null   str  
 3   severity         5795 non-null   str  
 4   message          5795 non-null   str  
 5   trace_id         5795 non-null   str  
 6   request_id       5795 non-null   str  
 7   method           5795 non-null   str  
 8   endpoint         5795 non-null   str  
 9   status_code      5795 non-null   int64
 10  latency_ms       5795 non-null   int64
 11  host             5795 non-null   str  
 12  env              5795 non-null   str  
 13  region           5795 non-null   str  
 14  log_type         5795 non-null   str  
dtypes: int64(2), str(13)
memory usage: 679.2 KB


In [5]:
df.head() # Devuelve las 5 primeras filas de cada columna

,timestamp_event,received_at,service_name,severity,message,trace_id,request_id,method,endpoint,status_code,latency_ms,host,env,region,log_type
0,2026-01-10T00:02:39.029160Z,2026-01-10T00:02:39.097160Z,orders-service,INFO,Request completed,bd8e6ebe717b4c43ba5e72e1668641a8,e6b3009adcad,GET,/orders/create,200,99,orders-service-pod-03,prod,sa-east-1,request
1,2026-01-10T00:02:46.081021Z,2026-01-10T00:02:46.196021Z,api-gateway,INFO,Background job completed,40ac5ff97bae43d5b8045484725e0aca,e2aa1cccd1cf,GET,/health,200,122,api-gateway-pod-01,prod,sa-east-1,request
2,2026-01-10T00:04:01.648849Z,2026-01-10T00:04:01.718849Z,notification-service,WARN,Rate limit nearing threshold,c048b378759a4c54a6f7d7251a6acc88,c74106ca8581,POST,/notify/sms,200,646,notification-service-pod-03,prod,sa-east-1,request
3,2026-01-10T00:05:08.148346Z,2026-01-10T00:05:08.236346Z,api-gateway,INFO,Background job completed,072169f708c84227986bfda9f9657bdc,bf10fa3609bd,GET,/checkout,200,127,api-gateway-pod-03,prod,sa-east-1,request
4,2026-01-10T00:05:19.590837Z,2026-01-10T00:05:19.632837Z,inventory-service,INFO,Health check OK,e76a214131d24a4ea1acf389362e34cb,f672d669eda7,GET,/inv/release,200,144,inventory-service-pod-02,prod,sa-east-1,request


In [6]:
df['timestamp_event'] = pd.to_datetime(df['timestamp_event']) # Convierto timestamp_event de string a formato datetime
df['window'] = df['timestamp_event'].dt.floor('5min')         # Creo mi ventana de 5min para el analisis de los datos
df['is_bad'] = (df['severity'].isin(['ERROR', 'CRITICAL']) | (df['status_code'] >= 500)) # Verifico eventos malos y los asigno a 'is_bad'

In [7]:
# - ¿Cuántos logs hay en total? → len(df)
# - ¿Qué severidad aparece más? → value_counts()
# - ¿Qué servicio genera más logs? → value_counts()
# - ¿Qué servicio genera menos logs? → value_counts().tail()
# - ¿Cuál es el mensaje más repetido? → value_counts().head(1)

print(f"En total existen {len(df)} logs registrados", '\n')
print("La severidad que mas se puede ver en el archivo es la de INFO:")
display(pd.DataFrame(df['severity'].value_counts()))
print("El servicio que genera mas logs segun la tabla es 'api-gateway', el que menos logs genera lo vemos al final como 'notification-service': ")
display(pd.DataFrame(df['service_name'].value_counts()))
print("El mensaje mas repetido es:")
display(pd.DataFrame(df['message'].value_counts().head(1)))

En total existen 5795 logs registrados 

La severidad que mas se puede ver en el archivo es la de INFO:


,count
severity,
INFO,3542
WARN,1358
ERROR,775
CRITICAL,120


El servicio que genera mas logs segun la tabla es 'api-gateway', el que menos logs genera lo vemos al final como 'notification-service': 


,count
service_name,
api-gateway,1509
orders-service,1057
inventory-service,964
payment-service,842
auth-service,778
notification-service,645


El mensaje mas repetido es:


,count
message,
Health check OK,1196


In [8]:
# - Un evento es malo si severity == 'ERROR' o 'CRITICAL' OR status_code >= 500
# - Crear columna booleana: df['is_bad'] = condicion
# - Imprimir cuántos hay en total

condicion = (df['severity'].isin(['ERROR', 'CRITICAL']) | (df['status_code'] >= 500)).astype(int)
df['is_bad'] = condicion
total_events = len(df)
bad_events = df['is_bad'].sum()
bad_rate = bad_events / total_events

print(total_events, bad_events, bad_rate)

5795 895 0.1544434857635893


In [9]:
pd.DataFrame(df['severity'].value_counts())

,count
severity,
INFO,3542
WARN,1358
ERROR,775
CRITICAL,120
